## 거시경제 지표에 따른 국내 이커머스 카테고리별 매출 탄력성 및 소비 전이 분석

- **용도:** 2026 국가데이터활용대회 공모전 참가 및 학생설계융합전공 졸업논문
  
- **분석 목적:** 거시경제 지표(물가, 금리, 유가)에 따른 국내 전자상거래 카테고리별 실질 거래액의 변동을 비교 및 예측하며, 소득 계층에 따른 소비 전이 현상을 파악
  
- **분석 활용 데이터:** KOSIS(온라인쇼핑동향조사, 품목별 소비자물가지수, 생활물가지수), ECOS(가계대출금리, 두바이유가, 경제심리지수), MDIS(가계동향조사)

- **2 Track 분석:**
    - Track 1. 22개 카테고리별 매출 탄력성 비교
    - Track 2. 소득 5분위별 소비 전이 효과 분석

In [60]:
# Ignore the warnings
import warnings
# warnings.filterwarnings('always')
warnings.filterwarnings('ignore')

# System related and data input controls
import os

# Data manipulation and visualization
import pandas as pd
pd.options.display.float_format = '{:,.2f}'.format
pd.options.display.max_rows = 20
pd.options.display.max_columns = 20
import numpy as np
import requests
from functools import reduce
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn import preprocessing

# 시각화 한글 폰트 깨짐 방지 (Windows 기준)
from matplotlib import font_manager, rc
font_path = "C:/Windows/Fonts/malgun.ttf"
font = font_manager.FontProperties(fname=font_path).get_name()
rc('font', family=font)

# Modeling algorithms
# General
import statsmodels.api as sm
from scipy import stats

# Model selection
from sklearn.model_selection import train_test_split

# Evaluation metrics
from sklearn import metrics

# Import Library: 분석에 사용할 모듈 설치
!pip install --upgrade pip
!python -m pip install --user --upgrade pip

In [61]:
df = pd.read_excel(r"C:\Users\sigtd\SJM\Data\온라인쇼핑동향조사_17.01~26.02.xlsx")

df.set_index('상품군별(1)', inplace=True)

df_transposed = df.T

date_strings = df_transposed.index.astype(str).str.replace('.', '')
df_transposed.index = pd.to_datetime(date_strings, format='%Y%m')

df_transposed.index.name = 'Date'

df_온라인거래액 = df_transposed.apply(pd.to_numeric, errors='coerce')

display(df_온라인거래액.head(3))
print(df_온라인거래액.info())

상품군별(1),합계,컴퓨터 및 주변기기,가전·전자·통신기기,서적,사무·문구,의복,신발,가방,패션용품 및 액세서리,스포츠·레저용품,...,생활용품,자동차 및 자동차용품,가구,애완용품,여행 및 교통서비스,문화 및 레저서비스,이쿠폰서비스,음식서비스,기타서비스,기타
Date,,,,,,,,,,,,,,,,,,,,,
2017-01-01,7310479,422384,602122,151230,53192,864515,103798,143643,143439,201380,...,592310,80144,188602,53096,1215463,164254,88648,183355,65597,144492
2017-02-01,7148849,475342,626874,154330,60403,827672,118631,163204,146885,207110,...,607119,76677,219176,62005,1110440,123892,80796,173924,40733,141578
2017-03-01,7747011,473351,667726,185571,70006,1042742,153293,168660,165677,257361,...,619493,81297,240688,65265,1072596,138133,90149,192095,35946,175648


<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 110 entries, 2017-01-01 to 2026-02-01
Data columns (total 24 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   합계           110 non-null    int64
 1   컴퓨터 및 주변기기   110 non-null    int64
 2   가전·전자·통신기기   110 non-null    int64
 3   서적           110 non-null    int64
 4   사무·문구        110 non-null    int64
 5   의복           110 non-null    int64
 6   신발           110 non-null    int64
 7   가방           110 non-null    int64
 8   패션용품 및 액세서리  110 non-null    int64
 9   스포츠·레저용품     110 non-null    int64
 10  화장품          110 non-null    int64
 11  아동·유아용품      110 non-null    int64
 12  음·식료품        110 non-null    int64
 13  농축수산물        110 non-null    int64
 14  생활용품         110 non-null    int64
 15  자동차 및 자동차용품  110 non-null    int64
 16  가구           110 non-null    int64
 17  애완용품         110 non-null    int64
 18  여행 및 교통서비스   110 non-null    int64
 19  문화 및 레저서비스   110 non-null    in

In [62]:
df_cpi_raw = pd.read_excel(r"C:\Users\sigtd\SJM\Data\소비자물가지수_17.01~26.02.xlsx")
df_cpi_raw.set_index(df_cpi_raw.columns[0], inplace=True)
df_cpi_t = df_cpi_raw.T

cleaned_dates = df_cpi_t.index.str.replace(' 월', '').str.replace('.', '-')
df_cpi_t.index = pd.to_datetime(cleaned_dates)
df_cpi_t.index.name = 'Date'

df_CPI = df_cpi_t.apply(pd.to_numeric, errors='coerce')

display(df_CPI.head(3))
print(df_CPI.info())

품목별,쌀,현미,찹쌀,보리쌀,콩,땅콩,혼식곡,배추,상추,시금치,...,미용료,뷰티미용료,산후조리원이용료,보험서비스료,자동차보험료,금융수수료,대입전형료,시험응시료,장례비,총지수
Date,,,,,,,,,,,,,,,,,,,,,
2017-01-01,70.07,77.75,78.22,104.43,86.06,88.87,87.17,83.47,76.07,85.94,...,93.16,93.88,94.42,89.62,92.32,100.09,114.79,94.62,97.57,97.37
2017-02-01,69.64,77.31,78.15,104.18,86.88,89.99,86.96,83.27,70.91,81.82,...,93.11,94.16,94.59,89.62,92.32,100.00,114.79,94.62,97.64,97.63
2017-03-01,68.56,76.25,76.09,103.58,86.16,89.32,85.31,81.34,66.67,61.41,...,93.47,94.24,94.70,89.62,93.06,100.00,114.79,94.62,97.52,97.56


<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 110 entries, 2017-01-01 to 2026-02-01
Columns: 459 entries, 쌀 to 총지수
dtypes: float64(459)
memory usage: 395.3 KB
None


## (종속변수 계산) 실질 온라인거래액
- 'KOSIS 온라인쇼핑동향조사'와 'KOSIS 소비자물가지수(CPI)' 데이터 활용
- '소비자물가지수(CPI)'의 458개 품목을 '온라인쇼핑동향조사'의 23개 카테고리와 매핑
- 매핑 완료 후 카테고리별로 명목 거래액을 대표 CPI로 나누어 물가 상승을 고려한 실질 거래액(종속변수) 계산

In [63]:
### CPI와 온라인거래액 매핑
# 1. 컬럼명 추출
# '합계'를 제외한 23개의 온라인 쇼핑 카테고리 추출
online_categories = df_온라인거래액.columns.drop('합계').tolist()
# 458개의 CPI 품목 추출
cpi_items = df_CPI.columns.tolist()

# 2. 카테고리별 매핑 딕셔너리 뼈대 생성
# Key: 온라인 카테고리명 / Value: 대표 CPI 품목명(리스트 형태)
category_mapping = {category: [] for category in online_categories}

category_mapping['컴퓨터 및 주변기기'] = ['컴퓨터', '저장장치', '컴퓨터소모품']
category_mapping['가전·전자·통신기기'] = ['휴대전화기', 'TV', '영상음향기기', '휴대용멀티미디어기기', '전기밥솥', '가스레인지', '전자레인지', '전기레인지', '냉장고', '김치냉장고', '에어컨', '선풍기', '공기청정기', '세탁기', '의류건조기', '식기세척기', '청소기', '소형주방가전', '헤어드라이어']
category_mapping['서적'] = ['유아용학습교재', '초등학교학습서', '중학교학습서', '고등학교학습서', '대학교재', '서적', '신문']
category_mapping['사무·문구'] = ['종이문구', '기타문구', '필기구', '회화용구']
category_mapping['의복'] = ['남자외의', '남자상의', '남자하의', '남자내의', '여자외의', '원피스', '여자상의', '여자하의', '여자내의', '점퍼', '티셔츠', '스웨터', '청바지', '운동복', '등산복', '유아동복', '양말']
category_mapping['신발'] = ['구두', '운동화', '실내화', '아동화']
category_mapping['가방'] = ['가방', '핸드백', '지갑', '우산']
category_mapping['패션용품 및 액세서리'] = ['손목시계', '장신구', '모자', '장갑', '선글라스']
category_mapping['스포츠·레저용품'] = ['자전거', '헬스기구', '레저용품', '운동용품']
category_mapping['화장품'] = ['기초화장품', '기능성화장품', '색조화장품', '모발염색약']
category_mapping['아동·유아용품'] = ['유모차', '장난감', '종이기저귀']

# 건강기능식품, 홍삼, 비타민제 등을 모두 음·식료품으로 통합
category_mapping['음·식료품'] = ['밀가루', '국수', '라면', '당면', '두부', '시리얼', '부침가루', '케이크', '빵', '떡', '파스타면', '소시지', '햄및베이컨', '기타육류가공품', '오징어채', '북어채', '어묵', '맛살', '수산물통조림', '젓갈', '우유', '분유', '치즈', '발효유', '참기름', '식용유', '과일가공품', '단무지', '맛김', '초콜릿', '사탕', '껌', '아이스크림', '비스킷', '스낵과자', '파이', '설탕', '잼', '물엿', '소금', '간장', '된장', '양념소스', '고추장', '카레', '식초', '드레싱', '혼합조미료', '스프', '이유식', '김치', '밑반찬', '냉동식품', '즉석식품', '편의점도시락', '삼각김밥', '커피', '차', '주스', '두유', '생수', '기능성음료', '탄산음료', '기타음료', '소주', '과실주', '맥주', '막걸리', '양주', '약주', '홍삼', '건강기능식품', '유산균', '비타민제']
category_mapping['농축수산물'] = ['쌀', '현미', '찹쌀', '보리쌀', '콩', '땅콩', '혼식곡', '배추', '상추', '시금치', '양배추', '미나리', '깻잎', '부추', '무', '열무', '당근', '감자', '고구마', '도라지', '콩나물', '버섯', '오이', '풋고추', '호박', '가지', '토마토', '파', '양파', '마늘', '브로콜리', '고사리', '파프리카', '생강', '사과', '배', '복숭아', '포도', '밤', '감', '귤', '오렌지', '참외', '수박', '딸기', '바나나', '키위', '블루베리', '망고', '체리', '아보카도', '파인애플', '아몬드', '고춧가루', '참깨', '인삼', '꿀', '국산쇠고기', '수입쇠고기', '돼지고기', '닭고기', '달걀', '갈치', '명태', '조기', '고등어', '오징어', '게', '굴', '조개', '전복', '새우', '마른멸치', '마른오징어', '낙지', '김', '미역']
category_mapping['생활용품'] = ['화초', '보온매트', '면도기', '침구', '커튼', '주택수선재료', '식기', '컵', '솥', '프라이팬', '냄비', '수저', '밀폐용기', '부엌용용구', '건전지', '소형가사용품', '세탁세제', '섬유유연제', '전구', '부엌용세제', '청소용세제', '살충제', '가정용비닐용품', '키친타월', '방향제', '습기제거제', '마스크', '반창고', '칫솔', '치약', '비누', '화장지', '구강세정제', '생리대', '샴푸', '바디워시']
category_mapping['자동차 및 자동차용품'] = ['소형승용차', '중형승용차', '대형승용차', '경승용차', '다목적승용차', '수입승용차', '전기동력차', '자동차용품', '자동차타이어', '블랙박스']
category_mapping['가구'] = ['장롱', '침대', '거실장', '소파', '책상', '의자', '식탁', '싱크대']

# 오프라인 미용 서비스 제외 (사료, 용품 등 실물 위주)
category_mapping['애완용품'] = ['반려동물용품'] 

# 교통/숙박 예약 위주로 남김 (시내버스료, 통행료 등 오프라인 성격 강한 항목 삭제)
category_mapping['여행 및 교통서비스'] = ['국제항공료', '국내항공료', '여객선료', '국내단체여행비', '해외단체여행비', '호텔숙박료', '여관숙박료', '콘도이용료', '휴양시설이용료', '열차료', '시외버스료', '승용차임차료']

# 온라인 예매가 중심이 되는 항목만 남김 (당구장, 목욕탕 등 오프라인 결제 항목 삭제)
category_mapping['문화 및 레저서비스'] = ['영화관람료', '공연예술관람료', '운동경기관람료', '관람시설이용료']

# 배달 비중이 높은 외식 항목 중심으로 재편성
category_mapping['음식서비스'] = ['치킨', '피자', '햄버거', '자장면', '짬뽕', '탕수육', '볶음밥', '돈가스', '김밥', '떡볶이', '쌀국수', '냉면', '칼국수', '생선초밥', '도시락', '커피(외식)']

# 렌탈 및 이사 등 용역서비스 중심 (공과금, 수리비 등 삭제)
category_mapping['기타서비스'] = ['가전제품렌탈비', '건강기기렌탈비', '이삿짐운송료', '사진서비스료', '가사도우미료', '택배이용료']

# 담배, 연료 등 온라인 거래 불가/희박 품목 삭제
category_mapping['기타'] = ['의료측정기', '보청기', '치료재료', '안경', '콘택트렌즈', '악기', '원예용품', '보일러', '감기약', '진통제', '소화제', '위장약', '진해거담제', '소염진통제', '피부질환제', '치과구강용약', '가정용비닐용품']

In [64]:
# 3. 실질 거래액(최종 종속변수 파생) 산출 함수
def calculate_real_volume(df_sales, df_cpi, mapping_dict):
    """
    온라인 거래액(명목)을 대표 CPI로 나누어 실질 거래액을 계산하는 함수
    """
    # 결과를 담을 새로운 데이터프레임 생성 (인덱스는 기존 거래액과 동일)
    df_real_volume = pd.DataFrame(index=df_sales.index)
    
    # 합계를 제외한 카테고리 순회
    for category in df_sales.columns.drop('합계'):
        
        # 딕셔너리에 해당 카테고리가 있고, 매핑된 CPI 품목이 1개 이상 존재하는 경우
        if category in mapping_dict and mapping_dict[category]:
            
            # 매핑된 CPI 품목들 (예: ['쌀', '배추'])
            mapped_items = mapping_dict[category]
            
            # 오타 방지를 위해 실제로 df_CPI에 존재하는 컬럼만 필터링
            valid_cpi_cols = [col for col in mapped_items if col in df_cpi.columns]
            
            if valid_cpi_cols:
                # 1. 대표 CPI 산출: 매핑된 품목이 여러 개면 평균을 대표값으로 사용
                representative_cpi = df_cpi[valid_cpi_cols].mean(axis=1)
                
                # 2. 실질 거래액 계산: 온라인 명목거래액 / (CPI 지수 / 100)
                # (물가지수는 2020년=100이 기준이므로 스케일을 맞추기 위해 100으로 나눔)
                df_real_volume[category] = df_sales[category] / (representative_cpi / 100)
            else:
                print(f"⚠️ 경고: '{category}'에 매핑된 {mapped_items}는 df_CPI에 존재하지 않는 컬럼입니다.")
                df_real_volume[category] = np.nan
        else:
            # 매핑된 품목이 비어있을 경우
            df_real_volume[category] = np.nan
            
    return df_real_volume

In [65]:
df_실질거래액 = calculate_real_volume(df_온라인거래액, df_CPI, category_mapping)
display(df_실질거래액.head())
print(df_실질거래액.info())

,컴퓨터 및 주변기기,가전·전자·통신기기,서적,사무·문구,의복,신발,가방,패션용품 및 액세서리,스포츠·레저용품,화장품,...,생활용품,자동차 및 자동차용품,가구,애완용품,여행 및 교통서비스,문화 및 레저서비스,이쿠폰서비스,음식서비스,기타서비스,기타
Date,,,,,,,,,,,,,,,,,,,,,
2017-01-01,"321,934.12","578,769.67","160,129.54","55,515.75","883,685.78","110,285.84","148,062.67","151,067.53","199,041.76","654,032.18",...,"594,288.81","78,719.27","211,710.76","51,954.05","1,246,271.88","177,665.05",NaN,"200,374.11","68,593.03","155,314.01"
2017-02-01,"400,304.85","597,791.45","162,889.86","63,057.07","846,031.38","123,104.62","164,557.07","154,602.36","205,233.14","728,001.32",...,"603,397.10","74,455.09","246,332.29","60,997.92","1,151,694.66","133,943.45",NaN,"189,387.63","42,488.33","152,170.31"
2017-03-01,"394,694.89","630,857.57","195,863.63","74,854.31","1,066,172.71","158,646.53","170,365.36","174,412.91","255,059.72","725,545.96",...,"615,009.94","78,833.54","270,630.95","64,821.62","1,125,849.63","150,059.34",NaN,"208,651.80","37,416.14","188,729.39"
2017-04-01,"306,437.79","681,756.74","129,207.87","64,603.72","994,531.97","163,964.70","166,584.63","173,019.03","280,343.56","578,498.31",...,"583,463.87","78,187.78","231,372.93","59,510.35","1,168,921.89","128,626.66",NaN,"212,979.94","55,365.46","175,135.03"
2017-05-01,"292,649.63","818,275.85","120,724.05","59,745.32","996,509.22","166,016.04","153,296.68","176,030.59","289,901.67","613,784.34",...,"604,639.85","84,736.23","223,672.49","56,320.39","1,206,662.76","143,633.21",NaN,"227,607.33","45,268.15","176,342.86"


<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 110 entries, 2017-01-01 to 2026-02-01
Data columns (total 23 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   컴퓨터 및 주변기기   110 non-null    float64
 1   가전·전자·통신기기   110 non-null    float64
 2   서적           110 non-null    float64
 3   사무·문구        110 non-null    float64
 4   의복           110 non-null    float64
 5   신발           110 non-null    float64
 6   가방           110 non-null    float64
 7   패션용품 및 액세서리  110 non-null    float64
 8   스포츠·레저용품     110 non-null    float64
 9   화장품          110 non-null    float64
 10  아동·유아용품      110 non-null    float64
 11  음·식료품        110 non-null    float64
 12  농축수산물        110 non-null    float64
 13  생활용품         110 non-null    float64
 14  자동차 및 자동차용품  110 non-null    float64
 15  가구           110 non-null    float64
 16  애완용품         110 non-null    float64
 17  여행 및 교통서비스   110 non-null    float64
 18  문화 및 레저서비스   110 non-null    fl

In [66]:
df_실질거래액 = df_실질거래액.drop('이쿠폰서비스', axis=1)

In [67]:
df_실질거래액

,컴퓨터 및 주변기기,가전·전자·통신기기,서적,사무·문구,의복,신발,가방,패션용품 및 액세서리,스포츠·레저용품,화장품,...,농축수산물,생활용품,자동차 및 자동차용품,가구,애완용품,여행 및 교통서비스,문화 및 레저서비스,음식서비스,기타서비스,기타
Date,,,,,,,,,,,,,,,,,,,,,
2017-01-01,"321,934.12","578,769.67","160,129.54","55,515.75","883,685.78","110,285.84","148,062.67","151,067.53","199,041.76","654,032.18",...,"269,785.81","594,288.81","78,719.27","211,710.76","51,954.05","1,246,271.88","177,665.05","200,374.11","68,593.03","155,314.01"
2017-02-01,"400,304.85","597,791.45","162,889.86","63,057.07","846,031.38","123,104.62","164,557.07","154,602.36","205,233.14","728,001.32",...,"173,994.65","603,397.10","74,455.09","246,332.29","60,997.92","1,151,694.66","133,943.45","189,387.63","42,488.33","152,170.31"
2017-03-01,"394,694.89","630,857.57","195,863.63","74,854.31","1,066,172.71","158,646.53","170,365.36","174,412.91","255,059.72","725,545.96",...,"207,094.84","615,009.94","78,833.54","270,630.95","64,821.62","1,125,849.63","150,059.34","208,651.80","37,416.14","188,729.39"
2017-04-01,"306,437.79","681,756.74","129,207.87","64,603.72","994,531.97","163,964.70","166,584.63","173,019.03","280,343.56","578,498.31",...,"194,646.38","583,463.87","78,187.78","231,372.93","59,510.35","1,168,921.89","128,626.66","212,979.94","55,365.46","175,135.03"
2017-05-01,"292,649.63","818,275.85","120,724.05","59,745.32","996,509.22","166,016.04","153,296.68","176,030.59","289,901.67","613,784.34",...,"197,912.04","604,639.85","84,736.23","223,672.49","56,320.39","1,206,662.76","143,633.21","227,607.33","45,268.15","176,342.86"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-10-01,"529,671.90","1,870,131.51","149,496.76","139,855.94","1,904,090.97","255,984.80","170,567.36","340,782.31","466,166.11","958,480.62",...,"899,098.51","1,346,991.20","512,896.34","351,790.88","208,649.42","2,631,015.16","251,976.17","2,775,856.33","129,082.04","215,410.67"
2025-11-01,"675,379.26","2,033,627.63","173,489.65","170,023.95","2,245,686.30","316,874.08","175,764.51","298,872.82","471,182.43","1,019,294.80",...,"959,300.16","1,404,752.70","680,887.78","384,862.48","207,672.73","2,613,196.63","233,903.83","2,724,077.43","140,373.51","238,951.14"
2025-12-01,"656,099.19","1,854,935.38","213,902.33","201,753.99","1,881,869.81","259,501.85","195,935.59","343,177.27","415,691.79","1,048,053.54",...,"1,018,714.03","1,394,438.19","642,001.23","371,069.67","213,741.17","2,694,956.26","274,633.00","2,955,223.41","141,750.81","233,950.73"


## (독립변수 추가) 실질 가계대출금리
- 'KOSIS 소비자물가지수(CPI)'와 'ECOS 예금은행 가계대출금리(잔액 기준)' 데이터 활용
- CPI의 총지수를 활용해 '(올해 총지수 - 작년 총지수) / 작년 총지수 × 100' 식으로 작년대비 물가상승률(YoY)을 계산
- 종속변수가 실질 거래액이므로 소비자의 실질적 구매력을 대변하는 '실질금리'만 사용 (명목 가계대출금리 - 물가상승률 = 실질 가계대출금리)

In [68]:
# df_CPI에서 총지수를 이용해 물가상승률 계산
# periods=12는 '12개월 전(즉, 작년 같은 달)'과 비교하라는 뜻입니다.
df_CPI['물가상승률(YoY)'] = df_CPI['총지수'].pct_change(periods=12) * 100

# 17년도 물가상승률은 16년도 데이터가 없으므로 직접 기입
values_2017 = [2.241, 2.083, 2.277, 1.956, 2.004, 1.806, 2.167, 2.489, 2.000, 1.762, 1.153, 1.407]

df_CPI.loc['2017-01-01':'2017-12-01', '물가상승률(YoY)'] = values_2017

# 결과 확인
display(df_CPI[['총지수', '물가상승률(YoY)']])

품목별,총지수,물가상승률(YoY)
Date,,
2017-01-01,97.37,2.24
2017-02-01,97.63,2.08
2017-03-01,97.56,2.28
2017-04-01,97.44,1.96
2017-05-01,97.55,2.00
...,...,...
2025-10-01,117.42,2.38
2025-11-01,117.20,2.45
2025-12-01,117.57,2.31


In [69]:
# ==============================================================================
# 1. ECOS 금리 데이터 불러오기 및 추출
# ==============================================================================
file_path_rate = r"C:\Users\sigtd\SJM\Data\예금은행 가계대출금리(잔액 기준)_17.01~26.02.xlsx"
df_rate_raw = pd.read_excel(file_path_rate)

# '계정항목' 컬럼에서 '가계대출'이 포함된 행만 필터링 (마이너스통장 포함 행)
# ECOS 파일 구조상 정확한 텍스트 매칭을 위해 str.contains를 사용합니다.
df_rate_filtered = df_rate_raw[df_rate_raw['계정항목'].str.contains('가계대출', na=False)]

# ==============================================================================
# 2. 시계열 전처리 (가로형 -> 세로형 및 날짜 인덱스 설정)
# ==============================================================================
# 가로로 퍼진 날짜(2017/01, 2017/02...)를 'Date'라는 하나의 열로 내립니다(Melt).
df_rate_melted = df_rate_filtered.melt(id_vars=['계정항목'], var_name='Date', value_name='명목_가계대출금리')

# '2017/01' 형태의 문자열을 완벽한 시계열(Datetime) 객체인 '2017-01-01'로 변환합니다.
df_rate_melted['Date'] = pd.to_datetime(df_rate_melted['Date'], format='%Y/%m')

# 날짜를 인덱스로 지정하고, 금리 데이터를 계산 가능한 숫자형으로 변환합니다.
df_rate_melted.set_index('Date', inplace=True)
df_rate_melted['명목_가계대출금리'] = pd.to_numeric(df_rate_melted['명목_가계대출금리'], errors='coerce')

# 최종적으로 병합에 사용할 금리 컬럼만 남깁니다.
df_rate_final = df_rate_melted[['명목_가계대출금리']]

# ==============================================================================
# 3. [핵심] 기존 데이터 결합 및 실질금리(최종 독립변수) 파생변수 생성
# ==============================================================================
# 기존에 물가상승률이 포함된 df_CPI 데이터와 날짜(Index)를 기준으로 병합합니다.
# how='inner'를 사용하여 두 데이터의 날짜가 완벽히 일치하는 구간만 남깁니다.
df_master = pd.merge(df_CPI, df_rate_final, left_index=True, right_index=True, how='inner')

# 피셔 방정식(Fisher Equation) 적용: 실질금리 = 명목금리 - 물가상승률
df_master['실질_가계대출금리'] = df_master['명목_가계대출금리'] - df_master['물가상승률(YoY)']

# ==============================================================================
# 4. 최종 결과 확인
# ==============================================================================
print("[데이터 결합 및 파생변수 생성 완료]")
print("현재 데이터 크기:", df_master.shape)
print("\n[최종 독립변수 데이터 미리보기]")
display(df_master[['물가상승률(YoY)', '명목_가계대출금리', '실질_가계대출금리']])

[데이터 결합 및 파생변수 생성 완료]
현재 데이터 크기: (110, 462)

[최종 독립변수 데이터 미리보기]


,물가상승률(YoY),명목_가계대출금리,실질_가계대출금리
Date,,,
2017-01-01,2.24,3.19,0.95
2017-02-01,2.08,3.21,1.13
2017-03-01,2.28,3.22,0.94
2017-04-01,1.96,3.23,1.27
2017-05-01,2.00,3.24,1.24
...,...,...,...
2025-10-01,2.38,4.31,1.93
2025-11-01,2.45,4.31,1.86
2025-12-01,2.31,4.31,2.00


In [70]:
df_master_final = df_실질거래액.join(df_master[['실질_가계대출금리']], how='inner')

## (독립변수 추가) 생활물가상승률
- 'KOSIS 생활물가지수' 데이터 활용
- 소비자물가지수(CPI)는 총 458개 품목 대상으로 국가 전체 인플레이션을 보여주지만, 생활물가지수는 구입 빈도가 높고 지출 비중이 높은 144개 필수 생필품만 모아 놓은 지수. 소비자 예산 제약과 체감경기 악화를 대변하는데 CPI보다 생활물가지수가 더 적합함. 풍선효과(선택재/사치재에서 필수재로의 품목 전이 현상)를 증명하는 핵심 변수임.
- 물가상승률과 마찬가지로 '(올해 총지수 - 작년 총지수) / 작년 총지수 × 100' 식으로 작년대비 물가상승률(YoY)을 계산함.

In [71]:
# 1. 데이터 로딩
df_living_raw = pd.read_excel(r"C:\Users\sigtd\SJM\Data\생활물가지수_16.01~26.02.xlsx")

# 2. 행렬 전치 및 시계열 인덱스 변환 (Wide -> Long 형태의 변형)
# '품목별' 열을 인덱스로 세팅하고 행과 열을 전치(.T)합니다.
df_living_raw.set_index('품목별', inplace=True)
df_living_cpi = df_living_raw.T

# 인덱스(날짜 문자열) 전처리: '2016.01' -> '2016-01-01' 형태로 시계열 변환
# 점(.)을 없애고 날짜형 객체로 바꿔줍니다.
df_living_cpi.index = df_living_cpi.index.str.replace('.', '')
df_living_cpi.index = pd.to_datetime(df_living_cpi.index, format='%Y%m')
df_living_cpi.index.name = 'Date'

# 안전한 계산을 위해 데이터 타입을 모두 실수형(float)으로 변환합니다.
df_living_cpi = df_living_cpi.apply(pd.to_numeric, errors='coerce')

# 3. 생활물가상승률(YoY) 파생변수 생성
# 첫 번째 열('생활물가지수' 총지수)을 지정하여 전년 동월 대비 증감률을 계산합니다.
target_col = df_living_cpi.columns[0]
df_living_cpi['생활물가상승률(YoY)'] = df_living_cpi[target_col].pct_change(periods=12) * 100

# 결합을 위해 필요한 파생변수 컬럼만 떼어냅니다.
df_living_feature = df_living_cpi[['생활물가상승률(YoY)']]

# 4. 기존 실질거래액 데이터셋과 최종 결합 (Merge)
# how='inner'를 통해 양쪽 데이터에 모두 존재하는 2017년 1월 ~ 2026년 2월 구간만 남깁니다.
df_master_final = df_master_final.join(df_living_feature, how='inner')

In [72]:
# 결과 확인
print("데이터 크기:", df_master_final.shape)
display(df_master_final.head())

데이터 크기: (110, 24)


,컴퓨터 및 주변기기,가전·전자·통신기기,서적,사무·문구,의복,신발,가방,패션용품 및 액세서리,스포츠·레저용품,화장품,...,자동차 및 자동차용품,가구,애완용품,여행 및 교통서비스,문화 및 레저서비스,음식서비스,기타서비스,기타,실질_가계대출금리,생활물가상승률(YoY)
Date,,,,,,,,,,,,,,,,,,,,,
2017-01-01,"321,934.12","578,769.67","160,129.54","55,515.75","883,685.78","110,285.84","148,062.67","151,067.53","199,041.76","654,032.18",...,"78,719.27","211,710.76","51,954.05","1,246,271.88","177,665.05","200,374.11","68,593.03","155,314.01",0.95,2.65
2017-02-01,"400,304.85","597,791.45","162,889.86","63,057.07","846,031.38","123,104.62","164,557.07","154,602.36","205,233.14","728,001.32",...,"74,455.09","246,332.29","60,997.92","1,151,694.66","133,943.45","189,387.63","42,488.33","152,170.31",1.13,2.50
2017-03-01,"394,694.89","630,857.57","195,863.63","74,854.31","1,066,172.71","158,646.53","170,365.36","174,412.91","255,059.72","725,545.96",...,"78,833.54","270,630.95","64,821.62","1,125,849.63","150,059.34","208,651.80","37,416.14","188,729.39",0.94,2.86
2017-04-01,"306,437.79","681,756.74","129,207.87","64,603.72","994,531.97","163,964.70","166,584.63","173,019.03","280,343.56","578,498.31",...,"78,187.78","231,372.93","59,510.35","1,168,921.89","128,626.66","212,979.94","55,365.46","175,135.03",1.27,2.44
2017-05-01,"292,649.63","818,275.85","120,724.05","59,745.32","996,509.22","166,016.04","153,296.68","176,030.59","289,901.67","613,784.34",...,"84,736.23","223,672.49","56,320.39","1,206,662.76","143,633.21","227,607.33","45,268.15","176,342.86",1.24,2.53


## (독립변수 추가) 유가상승률
- 'ECOS 국제상품가격_두바이유' 데이터 활용
- 한국은 필요한 원유의 약 70%를 중동 지역에서 수입하므로 미국 텍사스에서 생산되는 WTI나 유럽의 Brent유 가격변동보다 두바이유의 가격변동이 국내 정유사의 도입 단가에 직접적이고 절대적인 영향을 미침
- 전년동기대비 증감률로 유가상승률(YoY)을 독립변수로써 사용함

In [73]:
# 1. 유가상승률 데이터 로드
file_oil = r"C:\Users\sigtd\SJM\Data\유가상승률_17.01~26.02.xlsx"
df_oil_raw = pd.read_excel(file_oil)

# 2. Wide -> Long 변환
id_col_oil = df_oil_raw.columns[0]
df_oil_melted = df_oil_raw.melt(id_vars=[id_col_oil], var_name='Date', value_name='유가상승률(YoY)')

# 3. 날짜(Date) 전처리 및 인덱스 설정
df_oil_melted['Date'] = pd.to_datetime(df_oil_melted['Date'].astype(str).str.replace('/', '-'), format='%Y-%m')
df_oil_melted.set_index('Date', inplace=True)

# 4. 숫자형 변환 및 최종 피처 추출
df_oil_melted['유가상승률(YoY)'] = pd.to_numeric(df_oil_melted['유가상승률(YoY)'], errors='coerce')
df_oil_feature = df_oil_melted[['유가상승률(YoY)']].dropna()

# 결과 확인
print("✅ 유가상승률 전처리 완료")
display(df_oil_feature)

✅ 유가상승률 전처리 완료


,유가상승률(YoY)
Date,
2017-01-01,95.10
2017-02-01,83.00
2017-03-01,43.90
2017-04-01,33.60
2017-05-01,13.00
...,...
2025-10-01,-15.10
2025-11-01,-13.10
2025-12-01,-16.40


## (독립변수 추가) 경제심리지수(ESI) 순환변동치
- 지금까지의 독립변수(실질 가계대출금리, 생활물가상승률, 유가상승률)는 거시경제 영향으로 예산제약에 따른 물리적 타격이라면, 경제심리지수(ESI)는 심리적 타격을 대변함
- 기업과 소비자 모두를 포함한 민간의 경제상황에 대한 심리를 종합적으로 파악하기 위하여 BSI 및 CSI 지수를 합성하여 경제심리지수(ESI : Economic Sentiment Index)를 만듦
- 경제심리지수의 원계열, 순환변동치 중 순환변동치 데이터를 사용함
- '순환변동치'는 원계열에서 일시적인 충격이나 계절적 요인을 수학적으로 다 발라내고, "진짜 현재 경기가 바닥을 향해 가고 있는지, 회복세에 있는지"를 보여주는 순수한 트렌드 지표. 장기적인 거시경제 충격을 분석하는 논문의 목적에는 순환변동치가 완벽하게 부합함

In [74]:
# 1. ESI 데이터 로드
file_esi = r"C:\Users\sigtd\SJM\Data\경제심리지수(ESI)_순환변동치_17.01~26.02.xlsx"
df_esi_raw = pd.read_excel(file_esi)

# 2. Wide -> Long 변환 (첫 번째 열이 '계정항목' 등의 이름표라고 가정)
# df_esi_raw.columns[0]을 사용하여 첫 번째 열의 이름이 무엇이든 유연하게 대처합니다.
id_col_esi = df_esi_raw.columns[0]
df_esi_melted = df_esi_raw.melt(id_vars=[id_col_esi], var_name='Date', value_name='ESI_순환변동치')

# 3. 날짜(Date) 전처리 및 인덱스 설정
# '2017/01' 형태를 '2017-01-01' 형태의 시계열 객체로 변환
df_esi_melted['Date'] = pd.to_datetime(df_esi_melted['Date'].astype(str).str.replace('/', '-'), format='%Y-%m')
df_esi_melted.set_index('Date', inplace=True)

# 4. 숫자형 변환 및 최종 피처 추출
df_esi_melted['ESI_순환변동치'] = pd.to_numeric(df_esi_melted['ESI_순환변동치'], errors='coerce')
df_esi_feature = df_esi_melted[['ESI_순환변동치']].dropna()

# 결과 확인
print("✅ ESI 순환변동치 전처리 완료")
display(df_esi_feature)

✅ ESI 순환변동치 전처리 완료


,ESI_순환변동치
Date,
2017-01-01,98.20
2017-02-01,98.60
2017-03-01,99.10
2017-04-01,99.60
2017-05-01,100.10
...,...
2025-10-01,94.00
2025-11-01,94.40
2025-12-01,94.70


In [75]:
# 유가상승률 및 경제심리지수 독립변수 결합
df_master_final = df_master_final.join([df_oil_feature, df_esi_feature], how='inner')
display(df_master_final)
print(df_master_final.info())

,컴퓨터 및 주변기기,가전·전자·통신기기,서적,사무·문구,의복,신발,가방,패션용품 및 액세서리,스포츠·레저용품,화장품,...,애완용품,여행 및 교통서비스,문화 및 레저서비스,음식서비스,기타서비스,기타,실질_가계대출금리,생활물가상승률(YoY),유가상승률(YoY),ESI_순환변동치
Date,,,,,,,,,,,,,,,,,,,,,
2017-01-01,"321,934.12","578,769.67","160,129.54","55,515.75","883,685.78","110,285.84","148,062.67","151,067.53","199,041.76","654,032.18",...,"51,954.05","1,246,271.88","177,665.05","200,374.11","68,593.03","155,314.01",0.95,2.65,95.10,98.20
2017-02-01,"400,304.85","597,791.45","162,889.86","63,057.07","846,031.38","123,104.62","164,557.07","154,602.36","205,233.14","728,001.32",...,"60,997.92","1,151,694.66","133,943.45","189,387.63","42,488.33","152,170.31",1.13,2.50,83.00,98.60
2017-03-01,"394,694.89","630,857.57","195,863.63","74,854.31","1,066,172.71","158,646.53","170,365.36","174,412.91","255,059.72","725,545.96",...,"64,821.62","1,125,849.63","150,059.34","208,651.80","37,416.14","188,729.39",0.94,2.86,43.90,99.10
2017-04-01,"306,437.79","681,756.74","129,207.87","64,603.72","994,531.97","163,964.70","166,584.63","173,019.03","280,343.56","578,498.31",...,"59,510.35","1,168,921.89","128,626.66","212,979.94","55,365.46","175,135.03",1.27,2.44,33.60,99.60
2017-05-01,"292,649.63","818,275.85","120,724.05","59,745.32","996,509.22","166,016.04","153,296.68","176,030.59","289,901.67","613,784.34",...,"56,320.39","1,206,662.76","143,633.21","227,607.33","45,268.15","176,342.86",1.24,2.53,13.00,100.10
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-10-01,"529,671.90","1,870,131.51","149,496.76","139,855.94","1,904,090.97","255,984.80","170,567.36","340,782.31","466,166.11","958,480.62",...,"208,649.42","2,631,015.16","251,976.17","2,775,856.33","129,082.04","215,410.67",1.93,2.50,-15.10,94.00
2025-11-01,"675,379.26","2,033,627.63","173,489.65","170,023.95","2,245,686.30","316,874.08","175,764.51","298,872.82","471,182.43","1,019,294.80",...,"207,672.73","2,613,196.63","233,903.83","2,724,077.43","140,373.51","238,951.14",1.86,2.85,-13.10,94.40
2025-12-01,"656,099.19","1,854,935.38","213,902.33","201,753.99","1,881,869.81","259,501.85","195,935.59","343,177.27","415,691.79","1,048,053.54",...,"213,741.17","2,694,956.26","274,633.00","2,955,223.41","141,750.81","233,950.73",2.00,2.78,-16.40,94.70


<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 110 entries, 2017-01-01 to 2026-02-01
Data columns (total 26 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   컴퓨터 및 주변기기    110 non-null    float64
 1   가전·전자·통신기기    110 non-null    float64
 2   서적            110 non-null    float64
 3   사무·문구         110 non-null    float64
 4   의복            110 non-null    float64
 5   신발            110 non-null    float64
 6   가방            110 non-null    float64
 7   패션용품 및 액세서리   110 non-null    float64
 8   스포츠·레저용품      110 non-null    float64
 9   화장품           110 non-null    float64
 10  아동·유아용품       110 non-null    float64
 11  음·식료품         110 non-null    float64
 12  농축수산물         110 non-null    float64
 13  생활용품          110 non-null    float64
 14  자동차 및 자동차용품   110 non-null    float64
 15  가구            110 non-null    float64
 16  애완용품          110 non-null    float64
 17  여행 및 교통서비스    110 non-null    float64
 18  문화 및 레저서비스 

## 파생변수 추가 및 나머지 전처리

# 분석(Track 1): 카테고리별 매출 탄력성 비교
- 목적: 금리, 물가, 유가 등 거시 충격이 22개 상품군 중 어디를 가장 아프게 때리는가? 필수재(버티기) vs 선택재(무너짐)의 풍선효과 증명.